# 下載資料集和套件

In [1]:
import os
import cv2
import pandas as pd
import random
import shutil
from glob import glob

In [2]:
!pip install ultralytics -qq # YOLO套件

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.4 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
os.environ["KAGGLE_CONFIG_DIR"] = "/content/drive/MyDrive/.kaggle"

In [6]:
!kaggle competitions download -c taica-cvpdl-2025-hw-1 -p /content/drive/MyDrive/taica_hw1

taica-cvpdl-2025-hw-1.zip: Skipping, found more recently modified local copy (use --force to force download)


In [7]:
!cp /content/drive/MyDrive/taica_hw1/taica-cvpdl-2025-hw-1.zip .
!unzip -oq taica-cvpdl-2025-hw-1.zip -d /content/drive/MyDrive/taica_hw1

# 資料集前處理

In [8]:
img_dir = "/content/drive/MyDrive/taica_hw1/train/img"

files = os.listdir(img_dir)
print("共有", len(files), "張圖片")

img_path = os.path.join(img_dir, files[0])
img = cv2.imread(img_path)

print("img.shape =", img.shape)
h, w, c = img.shape
print(f"影像大小: {w}x{h}, 通道數: {c}")


共有 1266 張圖片
img.shape = (360, 640, 3)
影像大小: 640x360, 通道數: 3


## 切分資料集

In [ ]:
base_dir = "/content/drive/MyDrive/taica_hw1"
train_img_dir = os.path.join(base_dir, "train/img")
gt_path = os.path.join(base_dir, "train/gt.txt")


label_dir = os.path.join(base_dir, "train/labels")
os.makedirs(label_dir, exist_ok=True)


sample_img = cv2.imread(os.path.join(train_img_dir, os.listdir(train_img_dir)[0]))
H, W, C = sample_img.shape
print(f"影像大小: {W}x{H}, 通道數: {C}")


df = pd.read_csv(gt_path, header=None)
df.columns = ["image_id", "x", "y", "w", "h"]


for _, row in df.iterrows():
    image_id = row["image_id"]
    x, y, w, h = row["x"], row["y"], row["w"], row["h"]

    x_center = (x + w / 2) / W
    y_center = (y + h / 2) / H
    w_norm = w / W
    h_norm = h / H

    filename = f"{int(image_id):08d}.txt"
    label_path = os.path.join(label_dir, filename)

    with open(label_path, "a") as f:
        f.write(f"0 {x_center} {y_center} {w_norm} {h_norm}\n")

print("標註已轉換完成:", label_dir)

train_out = os.path.join(base_dir, "yolo_train")
val_out   = os.path.join(base_dir, "yolo_val")

for d in [train_out, val_out]:
    os.makedirs(os.path.join(d, "images"), exist_ok=True)
    os.makedirs(os.path.join(d, "labels"), exist_ok=True)

images = glob(os.path.join(train_img_dir, "*.jpg"))
random.shuffle(images)

split_idx = int(0.8 * len(images))
train_imgs, val_imgs = images[:split_idx], images[split_idx:]

def move_files(img_list, out_dir):
    for img_path in img_list:
        fname = os.path.basename(img_path)
        label_path = os.path.join(label_dir, fname.replace(".jpg", ".txt"))
        shutil.copy(img_path, os.path.join(out_dir, "images", fname))
        if os.path.exists(label_path):
            shutil.copy(label_path, os.path.join(out_dir, "labels", fname.replace(".jpg", ".txt")))

move_files(train_imgs, train_out)
move_files(val_imgs, val_out)
print(f"切分完成 Train: {len(train_imgs)}, Val: {len(val_imgs)}")


影像大小: 640x360, 通道數: 3
標註已轉換完成: /content/drive/MyDrive/taica_hw1/train/labels
切分完成 Train: 1012, Val: 254


## 影像增強

In [ ]:

import os, glob, shutil
from pathlib import Path
import random
import cv2
import numpy as np
from tqdm import tqdm
import albumentations as A


base_dir   = "/content/drive/MyDrive/taica_hw1"
train_out  = os.path.join(base_dir, "yolo_train")
val_out    = os.path.join(base_dir, "yolo_val")

img_dir    = os.path.join(train_out, "images")
lab_dir    = os.path.join(train_out, "labels")


img_out = os.path.join(train_out, "images_aug")
lab_out = os.path.join(train_out, "labels_aug")

os.makedirs(img_out, exist_ok=True)
os.makedirs(lab_out, exist_ok=True)


transform = A.Compose([
    A.RandomBrightnessContrast(p=0.7),
    A.HueSaturationValue(p=0.4),
    A.UnsharpMask(blur_limit=(3,3), alpha=(0.2,0.3), threshold=8, p=0.2),
    A.Resize(height=360, width=640)
], bbox_params=A.BboxParams(
    format="yolo",
    label_fields=["class_labels"],
    min_visibility=0.1
))


AUG_PER_IMG = 2

def load_yolo_labels(txt_path):

    bboxes, class_labels = [], []
    if not os.path.exists(txt_path):
        return bboxes, class_labels
    with open(txt_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                continue
            cls = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:])
            bboxes.append([xc, yc, w, h])
            class_labels.append(cls)
    return bboxes, class_labels

def save_yolo_labels(txt_path, bboxes, class_labels):

    with open(txt_path, "w") as f:
        for (xc, yc, w, h), cls in zip(bboxes, class_labels):
            f.write(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")


image_paths = sorted(glob.glob(os.path.join(img_dir, "*.jpg"))) + \
              sorted(glob.glob(os.path.join(img_dir, "*.png"))) + \
              sorted(glob.glob(os.path.join(img_dir, "*.jpeg")))

print(f"將對 {len(image_paths)} 張訓練影像做增強，每張生成 {AUG_PER_IMG} 份")

skipped = 0

for img_path in tqdm(image_paths):
    img = cv2.imread(img_path)
    if img is None:
        continue

    stem = Path(img_path).stem
    txt_path = os.path.join(lab_dir, f"{stem}.txt")
    bboxes, class_labels = load_yolo_labels(txt_path)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for k in range(AUG_PER_IMG):
        try:
            transformed = transform(image=img_rgb, bboxes=bboxes, class_labels=class_labels)

        except Exception as e:

            skipped += 1
            print(f"跳過 {Path(img_path).name} 的第 {k+1} 份增強：{e}")
            continue

        aug_img  = transformed["image"]
        aug_bxs  = transformed["bboxes"]
        aug_lbls = transformed["class_labels"]

        out_name = f"{stem}_aug{k+1:02d}"
        out_img_path = os.path.join(img_out, f"{out_name}.jpg")
        out_lab_path = os.path.join(lab_out, f"{out_name}.txt")

        cv2.imwrite(out_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
        save_yolo_labels(out_lab_path, aug_bxs, aug_lbls)

print(f" 影像增強完成，略過次數：{skipped}")


將對 1012 張訓練影像做增強，每張生成 2 份


 71%|███████   | 721/1012 [00:48<00:16, 18.18it/s]

跳過 00000911.jpg 的第 1 份增強：y_max is less than or equal to y_min for bbox [    0.29531     0.34167     0.29688     0.34167           0].
跳過 00000911.jpg 的第 2 份增強：y_max is less than or equal to y_min for bbox [    0.29531     0.34167     0.29688     0.34167           0].
跳過 00000912.jpg 的第 1 份增強：x_max is less than or equal to x_min for bbox [    0.29375     0.34444     0.29375     0.34444           0].
跳過 00000912.jpg 的第 2 份增強：x_max is less than or equal to x_min for bbox [    0.29375     0.34444     0.29375     0.34444           0].
跳過 00000915.jpg 的第 1 份增強：x_max is less than or equal to x_min for bbox [    0.29375     0.34167     0.29375     0.34167           0].
跳過 00000915.jpg 的第 2 份增強：x_max is less than or equal to x_min for bbox [    0.29375     0.34167     0.29375     0.34167           0].


100%|██████████| 1012/1012 [01:06<00:00, 15.16it/s]

 影像增強完成，略過次數：6


# 使用YOLO model


In [9]:
yaml_path = "/content/drive/MyDrive/taica_hw1/data.yaml"

with open(yaml_path, "w") as f:
    f.write(
        "train:\n"
        "  - /content/drive/MyDrive/taica_hw1/yolo_train/images\n"
        "  - /content/drive/MyDrive/taica_hw1/yolo_train/images_aug\n"
        "val: /content/drive/MyDrive/taica_hw1/yolo_val/images\n"
        "test: /content/drive/MyDrive/taica_hw1/test/images\n\n"
        "nc: 1\n"
        "names: ['pig']\n"
    )

print(' data.yaml 已更新完成！')


 data.yaml 已更新完成！


In [ ]:
!rm -f /content/drive/MyDrive/taica_hw1/yolo_train/images.cache
!rm -f /content/drive/MyDrive/taica_hw1/yolo_val/images.cache

## 訓練model

In [ ]:
model = YOLO("yolov10l.yaml")

model.train(
    data="/content/drive/MyDrive/taica_hw1/data.yaml",
    epochs=90,
    imgsz=640,
    batch=16,
    optimizer="adamw",
    cos_lr=True,
    patience=20,
    amp=True,
    weight_decay=0.05,
    pretrained=False,
    project="/content/drive/MyDrive/taica_hw1/runs",
)



Ultralytics 8.3.214 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/taica_hw1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=90, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10l.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=adamw, overlap_mask=True, patience=20, perspective=0.0, pl

KeyboardInterrupt: 

## 預測 test set

In [ ]:
model = YOLO("/content/drive/MyDrive/taica_hw1/runs/train/weights/best.pt")

model.predict(
    source="/content/drive/MyDrive/taica_hw1/test/img",
    imgsz=640,
    conf=0.1,
    save=True,
    save_txt=True,
    save_conf=True,
    project="/content/drive/MyDrive/taica_hw1/preds",
    name="test"
)



WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/1864 /content/drive/MyDrive/taica_hw1/test/img/00000001.jpg: 384x640 18 pigs, 115.4ms
image 2/1864 /content/drive/MyDrive/taica_hw1/test/img/00000002.jpg: 384x640 15 pigs, 33.9ms
image 3/1864 /content/drive/MyDrive/taica_hw1/test/img/00000003.jpg: 384x640 12 pigs, 33.7ms
image 4/1864 /content/drive/MyDrive/taica_hw1/test/img/00000004.jpg: 384x640 13 pigs, 33.7ms
image 5/1864 /content/drive/MyDrive/taica_hw1/test/img/00000005.jpg: 384x640 14 pig

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'pig'}
 obb: None
 orig_img: array([[[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [ 75,  75,  69],
         [ 75,  75,  69],
         [ 71,  71,  65]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [143, 143, 137],
         [144, 144, 138],
         [141, 141, 135]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [133, 133, 127],
         [133, 133, 127],
         [132, 132, 126]],
 
        ...,
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [191, 177, 178],
         [192, 176, 177],
         [ 25,   9,  10]],
 
        [[  0,   0,   0],
         [  0,   0,   0],
         [  0,   0,   0],
         ...,
         [174, 160, 1

# 將 labels 轉成 kaggle 競賽格式

In [ ]:
import os
import glob
import pandas as pd
from PIL import Image

IMG_W, IMG_H = 640, 360

pred_dir = "/content/drive/MyDrive/taica_hw1/preds/test/labels"
out_csv = "/content/drive/MyDrive/taica_hw1/submission.csv"

rows = []

for txt_file in sorted(glob.glob(os.path.join(pred_dir, "*.txt"))):
    img_id = os.path.splitext(os.path.basename(txt_file))[0]
    img_id = int(img_id)  # 轉成整數 ID (比賽要求)

    preds = []
    with open(txt_file, "r") as f:
        for line in f:
            cls, xc, yc, w, h, conf = map(float, line.strip().split())
            # YOLO 格式是 normalized，轉回像素
            x_left = (xc - w/2) * IMG_W
            y_top  = (yc - h/2) * IMG_H
            bw     = w * IMG_W
            bh     = h * IMG_H
            # 格式: conf x_left y_top width height class
            preds.append(f"{conf:.4f} {x_left:.0f} {y_top:.0f} {bw:.0f} {bh:.0f} {int(cls)}")

    pred_str = " ".join(preds)
    rows.append({"Image_ID": img_id, "PredictionString": pred_str})

df = pd.DataFrame(rows, columns=["Image_ID", "PredictionString"])
df.to_csv(out_csv, index=False)

print(f" Submission 檔案已存成: {out_csv}")
print(df.head(10))


 Submission 檔案已存成: /content/drive/MyDrive/taica_hw1/submission.csv
   Image_ID                                   PredictionString
0         1  0.2624 293 10 62 93 0 0.2113 494 0 86 55 0 0.2...
1         2  0.2425 493 0 85 54 0 0.1709 183 67 129 45 0 0....
2         3  0.3970 297 72 92 68 0 0.2453 495 -0 85 56 0 0....
3         4  0.2593 79 124 115 86 0 0.2371 494 0 85 56 0 0....
4         5  0.2632 495 -0 84 56 0 0.2442 80 123 113 86 0 0...
5         6  0.3097 299 74 89 67 0 0.2683 75 207 123 48 0 0...
6         7  0.2658 292 75 101 62 0 0.2232 186 69 122 44 0 ...
7         8  0.2281 225 139 110 59 0 0.2238 494 -0 85 56 0 ...
8         9  0.2700 225 139 108 58 0 0.2162 494 0 85 55 0 0...
9        10  0.2056 71 206 129 48 0 0.2031 494 0 85 54 0 0....
